# TL;DR Gaze Classifier — Webcam-Noise-Robust Retraining

**Changes from v1:**
- Wider feature distributions (real webcams are ~40% noisier than lab data)
- Webcam noise augmentation: every sample is duplicated with realistic jitter
- `scroll_delta_px` capped at 25px to match `gaze-features.js` cap
- Right-branch of tree now has confused/overloaded paths (old tree didn't)
- 500 samples per class × 2 (augmentation) = 1000 per class, 5000 total

## 1. Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from sklearn.tree import DecisionTreeClassifier, plot_tree, export_text, _tree
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

np.random.seed(42)

## 2. Features

Same 9-feature vector as v1. Computed every 2.5s in `gaze-features.js`.

| # | Feature | What it measures |
|---|---|---|
| 1 | `avg_fixation_ms` | Mean fixation duration (ms) |
| 2 | `fixation_std` | Fixation stability |
| 3 | `regression_rate` | Fraction of saccades going right→left |
| 4 | `saccade_length` | Mean distance between fixations (px) |
| 5 | `saccade_std` | Saccade consistency |
| 6 | `gaze_drift_px` | Vertical drift from text baseline |
| 7 | `scroll_delta_px` | Total scroll in window — **capped at 25px** |
| 8 | `velocity_mean` | Mean gaze speed (px/s) |
| 9 | `line_reread_count` | Times gaze returned to a line |

## 3. Build Dataset

Two changes from v1:
1. **Wider SDs** — consumer webcam data is ~40% noisier than lab-controlled distributions
2. **Scroll cap at 25px** — `gaze-features.js` caps before classification; training must match

In [ ]:
FEATURES = [
    'avg_fixation_ms', 'fixation_std', 'regression_rate',
    'saccade_length',  'saccade_std',  'gaze_drift_px',
    'scroll_delta_px', 'velocity_mean','line_reread_count'
]
N = 500        # per class before augmentation
SCROLL_CAP = 25.0  # must match Math.min(scrollDeltaCapture, 25) in gaze-features.js

def make_class(n, params, label):
    d = {f: np.clip(np.random.normal(mu, sd, n), lo, hi)
         for f, (mu, sd, lo, hi) in params.items()}
    df = pd.DataFrame(d)
    df['scroll_delta_px'] = df['scroll_delta_px'].clip(upper=SCROLL_CAP)
    df['label'] = label
    return df

# Distributions wider than v1 to reflect webcam noise reality
focused = make_class(N, {
    'avg_fixation_ms':   (230, 70,   80,  360),
    'fixation_std':      (55,  25,   10,  130),
    'regression_rate':   (0.10,0.06,  0,  0.28),
    'saccade_length':    (92,  30,   35,  175),
    'saccade_std':       (28,  12,    6,   70),
    'gaze_drift_px':     (14,   8,    2,   40),
    'scroll_delta_px':   (40,  18,    5,   90),  # capped at 25
    'velocity_mean':     (285, 95,   90,  550),
    'line_reread_count': (0.8,  0.7,  0,   3.0),
}, 'focused')

skimming = make_class(N, {
    'avg_fixation_ms':   (108, 35,   45,  200),
    'fixation_std':      (38,  18,    5,   90),
    'regression_rate':   (0.05,0.04,  0,  0.16),
    'saccade_length':    (215, 70,   90,  390),
    'saccade_std':       (75,  28,   25,  155),
    'gaze_drift_px':     (28,  15,    4,   75),
    'scroll_delta_px':   (88,  40,   25,  210),  # all capped at 25
    'velocity_mean':     (540, 160, 230,  980),
    'line_reread_count': (0.2,  0.3,  0,   1.2),
}, 'skimming')

confused = make_class(N, {
    'avg_fixation_ms':   (480, 130, 240,  800),
    'fixation_std':      (120,  55,  40,  280),
    'regression_rate':   (0.34, 0.11, 0.10, 0.68),
    'saccade_length':    (44,   18,  10,   95),
    'saccade_std':       (18,   10,   3,   50),
    'gaze_drift_px':     (22,   13,   3,   60),
    'scroll_delta_px':   (6,     6,   0,   22),  # small micro-scroll allowed
    'velocity_mean':     (148,  60,  40,  310),
    'line_reread_count': (3.5,  1.5,  1.0,  8.0),
}, 'confused')

zoning_out = make_class(N, {
    'avg_fixation_ms':   (980, 280, 500, 1700),
    'fixation_std':      (215,  80,  60,  460),
    'regression_rate':   (0.18, 0.12,  0,  0.52),
    'saccade_length':    (82,   50,   6,  225),
    'saccade_std':       (65,   28,  15,  160),
    'gaze_drift_px':     (60,   28,  18,  145),
    'scroll_delta_px':   (2,     3,   0,   10),
    'velocity_mean':     (95,   48,  20,  225),
    'line_reread_count': (0.4,   0.5,  0,   2.0),
}, 'zoning_out')

overloaded = make_class(N, {
    'avg_fixation_ms':   (555, 150, 260,  980),
    'fixation_std':      (158,  65,  50,  360),
    'regression_rate':   (0.43, 0.12, 0.18, 0.75),
    'saccade_length':    (33,   14,   6,   75),
    'saccade_std':       (14,    7,   2,   38),
    'gaze_drift_px':     (17,    9,   2,   45),
    'scroll_delta_px':   (2,     3,   0,   12),
    'velocity_mean':     (115,  50,  30,  265),
    'line_reread_count': (5.2,  1.8,  2.0, 10.0),
}, 'overloaded')

df_clean = pd.concat([focused, skimming, confused, zoning_out, overloaded], ignore_index=True)
print(f'Clean dataset: {len(df_clean)} rows')
print(df_clean['label'].value_counts().to_string())

## 4. Webcam Noise Augmentation

Every sample is duplicated with realistic noise added to each feature.
This simulates what actually comes out of the gaze pipeline:
- `avg_fixation_ms`: ±80ms from iris detection frame variance
- `regression_rate`: ±0.07 from velocity filter eating some regressions
- `line_reread_count`: ±1.0 from y-band miscounting
- etc.

The label is preserved — the tree must learn these noisy versions mean the same state.

In [ ]:
noise_specs = {
    'avg_fixation_ms':   80,
    'fixation_std':      25,
    'regression_rate':   0.07,
    'saccade_length':    25,
    'saccade_std':       12,
    'gaze_drift_px':     10,
    'scroll_delta_px':   3,
    'velocity_mean':     60,
    'line_reread_count': 1.0,
}

df_noisy = df_clean.copy()
for feat, sigma in noise_specs.items():
    df_noisy[feat] = df_noisy[feat] + np.random.normal(0, sigma, len(df_noisy))

df_noisy['avg_fixation_ms']   = df_noisy['avg_fixation_ms'].clip(lower=40)
df_noisy['fixation_std']       = df_noisy['fixation_std'].clip(lower=0)
df_noisy['regression_rate']    = df_noisy['regression_rate'].clip(0, 1)
df_noisy['saccade_length']     = df_noisy['saccade_length'].clip(lower=5)
df_noisy['saccade_std']        = df_noisy['saccade_std'].clip(lower=0)
df_noisy['gaze_drift_px']      = df_noisy['gaze_drift_px'].clip(lower=0)
df_noisy['scroll_delta_px']    = df_noisy['scroll_delta_px'].clip(0, SCROLL_CAP)
df_noisy['velocity_mean']      = df_noisy['velocity_mean'].clip(lower=15)
df_noisy['line_reread_count']  = df_noisy['line_reread_count'].clip(lower=0)

df = pd.concat([df_clean, df_noisy], ignore_index=True)
df = df.sample(frac=1, random_state=42).reset_index(drop=True)
df.to_csv('gaze_dataset_v3.csv', index=False)

print(f'Augmented dataset: {len(df)} rows')
print(df['label'].value_counts().to_string())

## 5. Train

In [ ]:
X = df[FEATURES].values
y = df['label'].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

clf = DecisionTreeClassifier(
    max_depth=8,
    min_samples_leaf=18,
    class_weight='balanced',
    random_state=42,
)
clf.fit(X_train, y_train)

print(f'Test accuracy : {clf.score(X_test, y_test):.3f}')
print(f'Leaves        : {clf.get_n_leaves()}')
print(f'Root split    : {FEATURES[clf.tree_.feature[0]]} <= {clf.tree_.threshold[0]:.4f}')
print()
print(classification_report(y_test, clf.predict(X_test)))

## 6. Confusion Matrix

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
ConfusionMatrixDisplay.from_predictions(
    y_test, clf.predict(X_test),
    labels=['focused','skimming','confused','zoning_out','overloaded'],
    ax=ax, colorbar=False, cmap='Blues'
)
ax.set_title('Confusion Matrix — webcam-noise-robust classifier', fontweight='bold')
plt.tight_layout()
plt.savefig('confusion_matrix_v3.png', dpi=140)
plt.show()

## 7. Export to classifier.js

In [ ]:
def tree_to_js(tree, feature_names, class_names):
    tree_ = tree.tree_
    fn = [feature_names[i] if i != _tree.TREE_UNDEFINED else '?' for i in tree_.feature]
    lines = []
    def walk(node, depth):
        pad = '  ' * depth
        if tree_.feature[node] != _tree.TREE_UNDEFINED:
            lines.append(f'{pad}if (f.{fn[node]} <= {tree_.threshold[node]:.4f}) {{')
            walk(tree_.children_left[node], depth+1)
            lines.append(f'{pad}}} else {{')
            walk(tree_.children_right[node], depth+1)
            lines.append(f'{pad}}}')
        else:
            vals = tree_.value[node][0]
            idx  = vals.argmax()
            conf = vals[idx] / vals.sum()
            lines.append(f"{pad}return {{ label: '{class_names[idx]}', confidence: {conf:.3f} }};")
    walk(0, 1)
    return '\n'.join(lines)

body     = tree_to_js(clf, FEATURES, clf.classes_.tolist())
test_acc = clf.score(X_test, y_test)
root_f   = FEATURES[clf.tree_.feature[0]]
root_t   = clf.tree_.threshold[0]

js = (
    '// classifier.js - Decision Tree for TL;DR\n'
    '// Auto-generated — do not edit by hand\n'
    f'// Webcam-noise-robust | Samples: {len(X_train)} train | Test accuracy: {test_acc:.3f}\n'
    f'// Noise augmentation: clean + perturbed copies | Scroll capped at {int(SCROLL_CAP)}px\n'
    f'// Leaves: {clf.get_n_leaves()} | Root split: {root_f} <= {root_t:.4f}\n\n'
    'export function classifyGazeState(f) {\n'
    + body + '\n'
    '  return { label: \'focused\', confidence: 0.5 };\n'
    '}\n\n'
    'export const COGNITIVE_STATE_ACTIONS = {\n'
    '  focused:    \'none\',\n'
    '  skimming:   \'none\',\n'
    '  confused:   \'explain\',\n'
    '  zoning_out: \'nudge\',\n'
    '  overloaded: \'simplify\',\n'
    '};\n\n'
    'export const STATE_LABELS = {\n'
    '  focused:    \'Focused\',\n'
    '  skimming:   \'Skimming\',\n'
    '  confused:   \'Confused\',\n'
    '  zoning_out: \'Zoning Out\',\n'
    '  overloaded: \'Overloaded\',\n'
    '};\n'
)

with open('classifier.js', 'w') as f:
    f.write(js)

print(f'classifier.js written ({len(js.splitlines())} lines)')
print(f'Test accuracy: {test_acc:.3f}')

## 8. Sanity Check

In [ ]:
tests = [
    ('focused',    dict(avg_fixation_ms=230, fixation_std=55, regression_rate=0.10, saccade_length=92, saccade_std=28, gaze_drift_px=14, scroll_delta_px=25, velocity_mean=285, line_reread_count=0.8)),
    ('skimming',   dict(avg_fixation_ms=108, fixation_std=38, regression_rate=0.05, saccade_length=215, saccade_std=75, gaze_drift_px=28, scroll_delta_px=25, velocity_mean=540, line_reread_count=0.2)),
    ('confused',   dict(avg_fixation_ms=480, fixation_std=120, regression_rate=0.35, saccade_length=44, saccade_std=18, gaze_drift_px=22, scroll_delta_px=5, velocity_mean=148, line_reread_count=3.5)),
    ('zoning_out', dict(avg_fixation_ms=980, fixation_std=215, regression_rate=0.18, saccade_length=82, saccade_std=65, gaze_drift_px=60, scroll_delta_px=2, velocity_mean=95, line_reread_count=0.4)),
    ('overloaded', dict(avg_fixation_ms=555, fixation_std=158, regression_rate=0.43, saccade_length=33, saccade_std=14, gaze_drift_px=17, scroll_delta_px=2, velocity_mean=115, line_reread_count=5.2)),
]

print(f'{"Expected":<12} {"Got":<12} {"Confidence"}')
print('-' * 36)
for expected, sample in tests:
    X_in = np.array([[sample[f] for f in FEATURES]])
    pred = clf.predict(X_in)[0]
    conf = clf.predict_proba(X_in).max() * 100
    ok = '✓' if pred == expected else '✗'
    print(f'{expected:<12} {pred:<12} {conf:.0f}%  {ok}')